# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omaradelahmed/fly-rank-internship1/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one content item (`content_hash_id`) within one client
(`client_hash_id`), summarized over a fixed feature window inside a single mid-panel month.
This is NOT one row per `report_date` — daily rows get aggregated up to the content level
before any modeling happens.

**Table(s):** `fact_content_daily_performance` (the `month=2026-03` partition only) for daily
search metrics, joined to `dim_content` for content metadata (`word_count`,
`content_updated_date`).

**Time window:** Decision point = `2026-03-16`.
- Feature window: `2026-03-01` → `2026-03-15` (strictly before the decision point)
- Label/proxy window: `2026-03-16` → `2026-03-31` (strictly after — used only to compute the
  outcome, never fed to a model)

**Label or proxy:** `is_declining` (proxy) = 1 if summed GSC impressions in the label window
are more than 20% lower than the feature window, else 0. Same rule as `trend_direction == "down"`
in the starter dataset, re-applied here on a fresh warehouse month instead of the starter CSV.

**One thing deliberately excluded:** `content_updated_date` values that fall AFTER the decision
point (`2026-03-16`) are excluded from the raw feature and treated as "unknown at decision
time" rather than used to compute `days_since_last_update` — a future update date cannot be
known at the moment the decision would actually be made.

## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id` | Context | pseudonymous IDs — grouping/joins/splits only, never model inputs |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (Mar 1–15) | Feature | observed strictly before the decision point |
| `word_count` | Feature | static content property, known before any decision point in this window |
| `days_since_last_update` (vs 2026-03-16, future dates nulled) | Feature | knowable at decision time once future-dated updates are excluded |
| `gsc_impressions`, `gsc_clicks` (Mar 16–31) | Label-only | used only to compute `is_declining`; never a model input |
| `is_declining` | Label | the target/proxy itself |
| `content_created_date` | Excluded | kept out of this 5-feature cap; not needed to prove the contract |
| `main_intent`, `content_type` | Excluded (here) | valid long-term features, left out only to respect "five features max" for this notebook |
| `provider_used`, `model_used` | Excluded | data dictionary explicitly flags these as "not a model feature" |
| `ga4_*` columns before a client's `ga4_data_start` | Excluded | zero-filled placeholders, not real zeros — must filter with `ga4_data_available IS TRUE` |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
%pip -q install duckdb

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_TABLE  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT  = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS  = f"read_parquet('{REL}/dim_clients.parquet')"

Paste your Hugging Face READ token (hf_...): ··········


In [13]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {MONTH_TABLE}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Duplicate (client, content, date) combos found: {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, date) combos found: 0


,client_hash_id,content_hash_id,report_date,c


In [14]:
counts_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {MONTH_TABLE}
""").df()

counts_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [15]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_with_ga4
    FROM {MONTH_TABLE}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,rows_with_ga4,pct_with_ga4
0,9841378,413966.0,4.2


In [17]:
import pandas as pd



features = con.sql(f"""
    WITH content_agg AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_feature_window,
            SUM(CASE WHEN report_date <= DATE '2026-03-16' THEN gsc_clicks ELSE 0 END)      AS clk_feature_window,
            AVG(CASE WHEN report_date <= DATE '2026-03-16' THEN gsc_avg_position END)       AS pos_feature_window,
            SUM(CASE WHEN report_date >  DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS imp_label_window
        FROM {MONTH_TABLE}
        GROUP BY 1, 2
        HAVING imp_feature_window >= 20
    )
    SELECT * FROM content_agg
""").df()

content_meta = con.sql(f"""
    SELECT content_hash_id, word_count,
           DATE_DIFF('day', content_updated_date, DATE '2026-03-16') AS days_since_last_update
    FROM {DIM_CONTENT}
""").df()

# future-dated updates aren't knowable at the decision point — see contract §1
content_meta.loc[content_meta['days_since_last_update'] < 0, 'days_since_last_update'] = pd.NA

features = features.merge(content_meta, on='content_hash_id', how='left')
features['is_declining'] = (features['imp_label_window'] < 0.8 * features['imp_feature_window']).astype(int)

print(f"Content items: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content items: 111,571


,client_hash_id,content_hash_id,imp_feature_window,clk_feature_window,pos_feature_window,imp_label_window,word_count,days_since_last_update,is_declining
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,58.0,0.0,3.555952,19.0,3579,NaN,1
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,212.0,2.0,4.068947,390.0,2455,NaN,0
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,501.0,1.0,4.429697,309.0,3653,NaN,1
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,58.0,0.0,6.593889,24.0,3096,NaN,1
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,828.0,2.0,1.846796,1030.0,3305,NaN,0


1. `imp_feature_window` — GSC impressions summed over Mar 1–15 only. **Knowable at the decision
   moment because** every value comes from `report_date <= 2026-03-16`.
2. `clk_feature_window` — same construction. **Knowable because** it only touches pre-decision
   dates.
3. `pos_feature_window` — average GSC position, Mar 1–15. **Knowable because** it's an observed
   daily metric restricted to the feature window.
4. `word_count` — static content property from `dim_content`. **Knowable because** it reflects
   the content as published, unaffected by the calendar.
5. `days_since_last_update` — days between `content_updated_date` and the decision point, with
   future dates nulled. **Knowable because** an update before 2026-03-16 was already on record
   by then; updates after are explicitly excluded, not assumed to be zero.

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

clean_cols = ['imp_feature_window', 'clk_feature_window', 'pos_feature_window',
              'word_count', 'days_since_last_update']

model_df = features.dropna(subset=clean_cols + ['is_declining']).copy()
X_clean, y = model_df[clean_cols], model_df['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y, test_size=0.25, random_state=42, stratify=y)
honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"Honest ROC-AUC (5 clean features only): {honest_auc:.3f}")

Honest ROC-AUC (5 clean features only): 0.594


In [19]:
total_rows = len(features)
usable_rows = len(model_df)  # after dropna

print(f"Total content items in the sample: {total_rows:,}")
print(f"Content items actually used to train the model (after dropna): {usable_rows:,}")
print(f"Fraction retained: {usable_rows/total_rows:.1%}")

Total content items in the sample: 111,571
Content items actually used to train the model (after dropna): 12,049
Fraction retained: 10.8%


In [20]:
# THE TRAP: imp_label_window is literally what is_declining is computed FROM — add it on purpose
leaky_cols = clean_cols + ['imp_label_window']
X_leaky = model_df[leaky_cols]

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, leaky_model.predict_proba(X_te_l)[:, 1])

print(f"Leaky ROC-AUC (with imp_label_window added): {leaky_auc:.3f}")
print(f"Jump from the leak: {leaky_auc - honest_auc:+.3f}")

Leaky ROC-AUC (with imp_label_window added): 1.000
Jump from the leak: +0.406


In [21]:
# Delete the leak, keep the honest number
del X_leaky, leaky_model
print(f"Kept score — honest ROC-AUC = {honest_auc:.3f} (no label-derived columns)")

Kept score — honest ROC-AUC = 0.594 (no label-derived columns)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [22]:
coverage_check = con.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {DIM_CLIENTS}) AS total_clients_in_warehouse,
        COUNT(DISTINCT client_hash_id) AS clients_active_in_march
    FROM {MONTH_TABLE}
""").df()

coverage_check

,total_clients_in_warehouse,clients_active_in_march
0,104,55


**Named limitation:** `month=2026-03` is one 31-day slice of a much longer, unbalanced panel
(2025-01-27 → 2026-06-30, ~17 months; per-client history depth varies from a few months to
12+). Only 55 of the warehouse's 104 clients were active in March — a pattern found in this
one month may reflect that month's seasonality or that specific subset of clients, not a
stable relationship.

The panel is also unbalanced *within* the month, not just across it: the availability check
in §3 shows only 4.2% of March rows have `ga4_data_available IS TRUE`. That's not a data
quality bug — most rows in this slice are GSC-only, either because the client's
`ga4_data_start` hadn't begun yet, or because GA4 simply wasn't wired up for most
client/content pairs during March. Any feature built from GA4 columns (sessions, engagement,
scroll rate) would silently apply to under 5% of the rows here unless explicitly filtered —
which is exactly why this notebook's five features stick to GSC and content-metadata fields
only, and why any future GA4-based feature needs its own coverage check before use.

This notebook proves the contract holds for one representative mid-panel month; it does not
claim the honest AUC above would replicate on a different month, on GA4-heavy clients, or on
the full table.

**A third limitation — severe sample shrinkage from missing values:** The 5-feature model
above drops any row with a missing value in any of its 5 columns before training
(`model_df = features.dropna(...)`). Because `days_since_last_update` is unknown for the
large majority of March content items at this decision point, the honest ROC-AUC of 0.593
was measured on only the fraction of content items shown in the cell above that survived the
drop — not the full 111,571-item sample. This surviving subset is systematically biased
toward content that happened to have a recorded update before 2026-03-16, which is very
likely not representative of the full population (it probably skews toward older, more
actively-maintained content and under-represents newer pages). A production version of this
feature should use explicit imputation-plus-flag handling (median/max fill + a
missing-indicator column, as done later in the full Lane 2 pipeline) instead of row-dropping —
precisely to avoid losing most of the sample to one sparse column.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.